# Individual and joint deterrence among LLM pricing agents: results review

This notebook calls the committed reporting scripts on frozen input folders. It makes no model calls.
Configure only this study below. Detailed diagnostics stay on disk; explicitly select figures for display.


## 1. Select the saved data

Fill `RUNS` with model-level folders. Empty experiment lists are omitted. Paths may be absolute
(e.g. a mounted Drive snapshot) or relative to `REPO`. Use a new `REPORT_ROOT` for every snapshot.
`MAIN_SEEDS` must match the saved plan; if models have different plans, run separate reports.


In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import pandas as pd
from IPython.display import Image, Markdown, display

REPO = Path.cwd()
if not (REPO / "pyproject.toml").is_file():
    REPO = REPO.parent
assert (REPO / "scripts/run_saved_report.py").is_file(), "Set REPO to the repository root"

RUNS = {
    "main": [],  # e.g. ["data/main/qwen38", "data/main/deepseek41"]
    "f1": [],
}
REPORT_ROOT = REPO / "reports/review-01"
LABEL = "Descriptive saved-data snapshot"
MAIN_STAGE = "pilot"  # pooled only when combining the documented continuation with stage 1
MAIN_SEEDS = [1, 2, 3, 4]


## 2. Rebuild tables and plots

Each experiment writes its own tables, figures and provenance manifest. Output folders cannot be
overwritten. Existing complete reports can be reopened without rerunning analysis; to regenerate,
choose a new report root. Failures remain visible rather than being relabelled as null effects.


In [ ]:
if not any(RUNS.values()):
    raise ValueError("Set at least one RUNS list to your saved model folders")
for kind, folders in RUNS.items():
    if not folders:
        continue
    out = REPORT_ROOT / kind
    if out.exists():
        manifest = json.loads((out / "report_manifest.json").read_text())
        expected = {str((REPO / p).resolve()) for p in folders}
        recorded = {str(Path(p).parent) for p in manifest["inputs"]}
        if expected != recorded:
            raise ValueError("Saved report inputs differ from RUNS; choose a new report root")
        if kind == "main" and (manifest["stage"] != MAIN_STAGE or manifest["planned_seeds"] != MAIN_SEEDS):
            raise ValueError("Saved Main report uses a different stage/seed plan; choose a new report root")
        if manifest["status"] != "complete":
            raise RuntimeError(f"{out} is incomplete; inspect its manifest and use a new report root")
        print(f"{kind}: opening existing report; no recomputation: {out}")
        continue
    command = [sys.executable, "-m", "scripts.run_saved_report", kind,
               *[str((REPO / p).resolve()) for p in folders], "--out", str(out), "--label", LABEL]
    if kind == "main":
        command += ["--stage", MAIN_STAGE, "--planned-seeds", *map(str, MAIN_SEEDS)]
    result = subprocess.run(command, cwd=REPO, capture_output=True, text=True)
    REPORT_ROOT.mkdir(parents=True, exist_ok=True)
    log = REPORT_ROOT / f"{kind}.report.log"
    log.write_text(result.stdout + result.stderr)
    if result.returncode:
        print("\n".join(log.read_text().splitlines()[-30:]))
        raise RuntimeError(f"{kind} report failed; full log: {log}")
    print(f"{kind}: tables and figures ready in {out}")


## 3. Check denominators before interpreting effects

Main: count complete eligible games separately from ineligible and incomplete games.
F1: count complete paired branches separately from calm checkpoints and completed parents.
F2: coverage is not accuracy; two judgments from one case are not independent samples.

The preview below is capped at 20 rows. Open the saved CSV for the full table.


In [ ]:
for kind, folders in RUNS.items():
    if not folders:
        continue
    folder = REPORT_ROOT / kind
    manifest = json.loads((folder / "report_manifest.json").read_text())
    display(Markdown(f"### {kind.upper()} — {manifest['label']}"))
    print("Created:", manifest["created_utc"], "| status:", manifest["status"])
    filenames = {
        "main": ["accounting.csv", "primary.csv"],
        "f1": ["replacement_games.csv", "replacement_summary.csv"],
        "f2": ["accounting.csv", "summary.csv"],
    }[kind]
    for name in filenames:
        path = folder / "tables" / name
        if not path.exists():
            print(name, "not available")
            continue
        try:
            table = pd.read_csv(path)
        except pd.errors.EmptyDataError:
            print(name, "has no rows; this is not a zero effect")
            continue
        print(f"{name}: {len(table)} rows; {path}")
        display(table.head(20))


## 4. Review a deliberate figure selection

`figure_index.csv` lists all saved PNGs without displaying them. Select figures by the question they
answer and coverage of the planned models—not by effect size or significance. Keep one clear figure
for a claim; use a table when exact counts are easier to read.

Suggested paper figures:

- **Main + F1:** one design diagram; Main gain/paired-contrast panels with eligible n; F1 primary
  removal contrast with complete-pair n. Add price paths only if they explain a finding the effects
  plot cannot. With very few games, show all eligible games rather than a selected impressive trace.
- **F2:** one evidence-design diagram; primary unsupported-positive and resolved-accuracy panels,
  accompanied by coverage and conditional denominators. A compact secondary panel/table should
  show unsupported negatives and same-case baseline comparisons when they change the interpretation.
- **Supplement:** complete per-game trajectories, controls, additional metrics and diagnostics.

No plot is required merely because the code can produce it. The shortlist is a starting point,
not a ceiling that prevents showing scientifically necessary evidence.


In [ ]:
figure_paths = sorted(REPORT_ROOT.glob("*/figures/**/*.png"))
index = pd.DataFrame({"figure": [str(p.relative_to(REPORT_ROOT)) for p in figure_paths]})
index.to_csv(REPORT_ROOT / "figure_index.csv", index=False)
print(f"{len(index)} figures saved; none displayed automatically. Full index: {REPORT_ROOT / 'figure_index.csv'}")
display(index.head(12))


Choose a few files from the index below. For each, write the question and a cautious reading.
Do not paste a whole directory glob here. Six inline figures is a readability guard, not a limit on
saved outputs or evidence in the supplement.


In [ ]:
SELECTED_FIGURES = [
    # {"path": "main/figures/main_primary_<hash>.png",
    #  "question": "How does the cut gain change when one versus both rivals respond?",
    #  "reading": "State eligible n and paired S−J; unavailable intervals are not zero effects."},
]
if len(SELECTED_FIGURES) > 6:
    raise ValueError("Keep this review to six figures; inspect additional diagnostics separately")
for item in SELECTED_FIGURES:
    path = (REPORT_ROOT / item["path"]).resolve()
    if not path.is_relative_to(REPORT_ROOT.resolve()) or not path.is_file():
        raise ValueError(f"Invalid figure path: {path}")
    display(Markdown(f"**Question:** {item['question']}\n\n**Reading:** {item['reading']}"))
    display(Image(filename=str(path), width=850))


## 5. Turn figures into findings

Write one short paragraph per central result:

1. **Observation:** effect/rate, units, applicable denominator, uncertainty and snapshot.
2. **Interpretation:** answer the research question at the scope supported by those numbers.
3. **Limit:** explain the important ambiguity, selection or missingness.

Do not turn a non-significant estimate into evidence of no effect. A/A describes repeatability in
one condition. F1 does not identify awareness or habit. F2 unsupported negatives are unjustified
conclusions on ambiguous evidence, not necessarily false negatives against hidden truth.

Before submitting: reconcile final counts; replace interim labels; choose readable panels and captions;
check the PDF; retain scripts, CSVs, manifests and the exact code release alongside the paper.
